# Prompt-injection benchmark visualizations

Edit `RUN_DIRECTORIES` in the first code cell to select benchmark versions. Incomplete runs are reported and excluded automatically. Every plotting cell saves one figure as PNG and PDF.

In [ ]:
from __future__ import annotations

import json
import math
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "attack" / "output").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing attack/output")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# User configuration: add, remove, or rename benchmark directories here.
RUN_DIRECTORIES = {
    "V0": PROJECT_ROOT / "attack/output/v0",
    "V1": PROJECT_ROOT / "attack/output/v1",
    "V2": PROJECT_ROOT / "attack/output/v2",
    "V3": PROJECT_ROOT / "attack/output/v3",
}
FIGURE_DIRECTORY = PROJECT_ROOT / "attack/output/figures"
SAVE_FORMATS = ("png", "pdf")
FIGURE_DPI = 300

ATTACK_ORDER = ["naive", "escape_characters", "context_ignoring", "fake_completion", "combined"]
TASK_ORDER = ["sentiment", "spam", "duplicate", "hate", "nli"]
ATTACK_LABELS = {
    "naive": "Naive",
    "escape_characters": "Escape characters",
    "context_ignoring": "Context ignoring",
    "fake_completion": "Fake completion",
    "combined": "Combined",
}
TASK_LABELS = {
    "sentiment": "Sentiment",
    "spam": "Spam",
    "duplicate": "Duplicate",
    "hate": "Hate",
    "nli": "NLI",
}

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.frameon": False,
})

FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Figures will be saved to: {FIGURE_DIRECTORY}")

In [ ]:
def read_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        return None


def read_jsonl(path: Path) -> list[dict]:
    if not path.is_file():
        return []
    rows = []
    try:
        for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            if line.strip():
                value = json.loads(line)
                if isinstance(value, dict):
                    rows.append(value)
    except (OSError, json.JSONDecodeError):
        return []
    return rows


def is_number(value) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def ordered_categories(found: set[str], preferred: list[str]) -> list[str]:
    return [item for item in preferred if item in found] + sorted(found - set(preferred))


def load_run(version: str, directory: Path) -> dict:
    directory = Path(directory)
    config = read_json(directory / "run_config.json")
    metrics = read_json(directory / "metrics.json")
    cases = read_jsonl(directory / "cases.jsonl")
    messages = []

    metric_rows = metrics.get("attacks", []) if isinstance(metrics, dict) else []
    metric_rows = [row for row in metric_rows if isinstance(row, dict)]
    configured_attacks = config.get("attacks", []) if isinstance(config, dict) else []
    configured_tasks = [item.get("name") for item in config.get("tasks", []) if isinstance(item, dict)] if isinstance(config, dict) else []
    configured_attacks = [item for item in configured_attacks if isinstance(item, str)]
    configured_tasks = [item for item in configured_tasks if isinstance(item, str)]

    expected_groups = {(attack, task) for attack in configured_attacks for task in configured_tasks}
    observed_groups = {(row.get("attack_method"), row.get("injected_task")) for row in metric_rows}
    required_metric_fields = ("asv", "matching_rate", "target_accuracy_drop")
    numeric_metrics = bool(metric_rows) and all(
        all(is_number(row.get(field)) for field in required_metric_fields)
        for row in metric_rows
    )
    metrics_complete = bool(config) and bool(metrics) and bool(expected_groups) and expected_groups == observed_groups and numeric_metrics

    if not directory.is_dir():
        messages.append("directory missing")
    elif not isinstance(config, dict):
        messages.append("run_config.json missing or invalid")
    elif not isinstance(metrics, dict):
        messages.append("metrics.json missing or invalid")
    elif expected_groups != observed_groups:
        messages.append(f"incomplete metric grid ({len(observed_groups)}/{len(expected_groups)} groups)")
    elif not numeric_metrics:
        messages.append("required metrics are missing or non-numeric")

    sample_size = config.get("sample_size_per_task") if isinstance(config, dict) else None
    expected_cases = sample_size * len(configured_tasks) * len(configured_attacks) if isinstance(sample_size, int) else None
    cases_complete = metrics_complete and expected_cases is not None and len(cases) == expected_cases
    if metrics_complete and not cases_complete:
        messages.append(f"incomplete cases.jsonl ({len(cases)}/{expected_cases})")

    return {
        "version": version,
        "directory": directory,
        "config": config or {},
        "metrics": metrics or {},
        "metric_rows": metric_rows,
        "cases": cases,
        "metrics_complete": metrics_complete,
        "cases_complete": cases_complete,
        "expected_cases": expected_cases,
        "messages": messages,
    }


def mean_metric(run: dict, group_field: str, group_value: str, metric: str) -> float:
    values = [
        row[metric] for row in run["metric_rows"]
        if row.get(group_field) == group_value and is_number(row.get(metric))
    ]
    return statistics.fmean(values) if values else float("nan")


def version_colors(versions: list[str]) -> dict[str, object]:
    cmap = plt.get_cmap("tab10")
    return {version: cmap(index % 10) for index, version in enumerate(versions)}


def save_figure(fig, stem: str) -> None:
    saved = []
    for extension in SAVE_FORMATS:
        path = FIGURE_DIRECTORY / f"{stem}.{extension}"
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
        saved.append(path.name)
    print("Saved:", ", ".join(saved))


def grouped_bars(ax, categories, values_by_version, colors, ylabel, title, scale=1.0, value_format="{:.2f}"):
    versions = list(values_by_version)
    x = np.arange(len(categories), dtype=float)
    width = 0.82 / max(len(versions), 1)
    offsets = (np.arange(len(versions)) - (len(versions) - 1) / 2) * width
    all_values = []
    for offset, version in zip(offsets, versions):
        values = np.asarray(values_by_version[version], dtype=float) * scale
        all_values.extend(values[np.isfinite(values)])
        bars = ax.bar(x + offset, values, width=width, label=version, color=colors[version])
        labels = [value_format.format(value) if np.isfinite(value) else "" for value in values]
        ax.bar_label(bars, labels=labels, padding=2, fontsize=8, rotation=90 if len(versions) > 4 else 0)
    ax.set_xticks(x, categories)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(ncol=min(len(versions), 4), loc="upper left")
    ax.grid(axis="x", visible=False)
    if all_values:
        lower = min(0.0, min(all_values) * 1.15)
        upper = max(all_values)
        ax.set_ylim(lower, max(scale, upper * 1.18) if scale == 1.0 else upper * 1.18)


runs = [load_run(version, Path(directory)) for version, directory in RUN_DIRECTORIES.items()]
metric_runs = [run for run in runs if run["metrics_complete"]]
case_runs = [run for run in runs if run["cases_complete"]]
colors = version_colors([run["version"] for run in metric_runs])

summary_rows = []
for run in runs:
    config = run["config"]
    summary_rows.append([
        run["version"],
        str(run["directory"]),
        config.get("variant", "—"),
        config.get("model", "—"),
        "yes" if run["metrics_complete"] else "no",
        f"{len(run['cases'])}/{run['expected_cases'] or 'unknown'}",
        "; ".join(run["messages"]) or "ready",
    ])

headers = ["Version", "Directory", "Variant", "Model", "Metrics ready", "Cases", "Status"]
widths = [max(len(str(row[i])) for row in [headers, *summary_rows]) for i in range(len(headers))]
print(" | ".join(header.ljust(widths[i]) for i, header in enumerate(headers)))
print("-+-".join("-" * width for width in widths))
for row in summary_rows:
    print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))

In [ ]:
# Graph 1: ASV by attack method
if not metric_runs:
    print("No complete benchmark directories are available for this graph.")
else:
    found = {row.get("attack_method") for run in metric_runs for row in run["metric_rows"] if row.get("attack_method")}
    attacks = ordered_categories(found, ATTACK_ORDER)
    values = {run["version"]: [mean_metric(run, "attack_method", attack, "asv") for attack in attacks] for run in metric_runs}
    fig, ax = plt.subplots(figsize=(11, 5.8))
    grouped_bars(ax, [ATTACK_LABELS.get(a, a) for a in attacks], values, colors, "Mean ASV", "Attack success by attack method")
    ax.set_ylim(0, 1.08)
    fig.tight_layout()
    save_figure(fig, "01-asv-by-attack-method")
    plt.show()

In [ ]:
# Graph 2: ASV versus matching rate
if not metric_runs:
    print("No complete benchmark directories are available for this graph.")
else:
    found = {row.get("attack_method") for run in metric_runs for row in run["metric_rows"] if row.get("attack_method")}
    attacks = ordered_categories(found, ATTACK_ORDER)
    markers = ["o", "s", "^", "D", "P", "X", "v", "<", ">"]
    attack_markers = {attack: markers[index % len(markers)] for index, attack in enumerate(attacks)}
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.2, color="0.45", label="ASV = matching rate")
    for run in metric_runs:
        for attack in attacks:
            asv = mean_metric(run, "attack_method", attack, "asv")
            matching = mean_metric(run, "attack_method", attack, "matching_rate")
            if np.isfinite(asv) and np.isfinite(matching):
                ax.scatter(matching, asv, s=75, marker=attack_markers[attack], color=colors[run["version"]], edgecolor="white", linewidth=0.7)
    version_handles = [Line2D([0], [0], marker="o", linestyle="", markerfacecolor=colors[run["version"]], markeredgecolor="none", label=run["version"]) for run in metric_runs]
    attack_handles = [Line2D([0], [0], marker=attack_markers[attack], linestyle="", color="0.25", label=ATTACK_LABELS.get(attack, attack)) for attack in attacks]
    legend_versions = ax.legend(handles=version_handles, title="Version", loc="lower right")
    ax.add_artist(legend_versions)
    ax.legend(handles=attack_handles, title="Attack", loc="upper left")
    ax.set(xlim=(0, 1.02), ylim=(0, 1.02), xlabel="Mean matching rate", ylabel="Mean ASV", title="Attack success versus behavioral matching")
    ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    save_figure(fig, "02-asv-vs-matching-rate")
    plt.show()

In [ ]:
# Graph 3: ASV by version and injected task
if not metric_runs:
    print("No complete benchmark directories are available for this graph.")
else:
    found = {row.get("injected_task") for run in metric_runs for row in run["metric_rows"] if row.get("injected_task")}
    tasks = ordered_categories(found, TASK_ORDER)
    matrix = np.asarray([[mean_metric(run, "injected_task", task, "asv") for task in tasks] for run in metric_runs], dtype=float)
    fig_width = max(7.2, 1.35 * len(tasks) + 2.5)
    fig_height = max(3.2, 0.72 * len(metric_runs) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    image = ax.imshow(np.ma.masked_invalid(matrix), vmin=0, vmax=1, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(tasks)), [TASK_LABELS.get(task, task) for task in tasks])
    ax.set_yticks(range(len(metric_runs)), [run["version"] for run in metric_runs])
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[row_index, column_index]
            if np.isfinite(value):
                text_color = "white" if value >= 0.58 else "black"
                ax.text(column_index, row_index, f"{value:.2f}", ha="center", va="center", color=text_color, fontweight="bold")
    colorbar = fig.colorbar(image, ax=ax, pad=0.02)
    colorbar.set_label("Mean ASV")
    ax.set_xlabel("Injected task")
    ax.set_ylabel("Pipeline version")
    ax.set_title("Attack success by pipeline version and injected task")
    fig.tight_layout()
    save_figure(fig, "03-asv-by-version-and-task")
    plt.show()

In [ ]:
# Graph 4: target accuracy drop by attack method
if not metric_runs:
    print("No complete benchmark directories are available for this graph.")
else:
    found = {row.get("attack_method") for run in metric_runs for row in run["metric_rows"] if row.get("attack_method")}
    attacks = ordered_categories(found, ATTACK_ORDER)
    values = {run["version"]: [mean_metric(run, "attack_method", attack, "target_accuracy_drop") for attack in attacks] for run in metric_runs}
    fig, ax = plt.subplots(figsize=(11, 5.8))
    grouped_bars(ax, [ATTACK_LABELS.get(a, a) for a in attacks], values, colors, "Mean target accuracy drop (percentage points)", "Damage to target-task accuracy by attack method", scale=100.0, value_format="{:.1f}")
    ax.axhline(0, color="0.35", linewidth=0.9)
    fig.tight_layout()
    save_figure(fig, "04-target-accuracy-drop")
    plt.show()

In [ ]:
# Graph 5: computational overhead by version
if not case_runs:
    print("No complete benchmark directories with usable cases.jsonl are available for this graph.")
else:
    overhead = []
    for run in case_runs:
        usable = []
        for row in run["cases"]:
            usage = row.get("token_usage")
            latency = row.get("latency_ms")
            if row.get("error") is None and is_number(latency) and isinstance(usage, dict):
                input_tokens = usage.get("input")
                output_tokens = usage.get("output")
                if is_number(input_tokens) and is_number(output_tokens):
                    usable.append((latency / 1000.0, input_tokens + output_tokens))
        if usable:
            overhead.append({
                "version": run["version"],
                "median_latency": statistics.median(item[0] for item in usable),
                "mean_tokens": statistics.fmean(item[1] for item in usable),
                "count": len(usable),
            })

    if not overhead:
        print("Complete case files were found, but none contained usable latency and token measurements.")
    else:
        labels = [f"{item['version']}\n(n={item['count']:,})" for item in overhead]
        bar_colors = [colors.get(item["version"], plt.get_cmap("tab10")(index % 10)) for index, item in enumerate(overhead)]
        fig, axes = plt.subplots(1, 2, figsize=(11, 5))
        latency_bars = axes[0].bar(labels, [item["median_latency"] for item in overhead], color=bar_colors)
        axes[0].bar_label(latency_bars, fmt="%.2f s", padding=3)
        axes[0].set(title="Median latency per attacked case", ylabel="Seconds")
        token_bars = axes[1].bar(labels, [item["mean_tokens"] for item in overhead], color=bar_colors)
        axes[1].bar_label(token_bars, fmt="%.0f", padding=3)
        axes[1].set(title="Mean tokens per attacked case", ylabel="Input + output tokens")
        for ax in axes:
            ax.set_xlabel("Pipeline version")
            ax.grid(axis="x", visible=False)
            ax.margins(y=0.16)
        fig.suptitle("Computational overhead of prompt-injection evaluation", fontsize=14)
        fig.tight_layout()
        save_figure(fig, "05-computational-overhead")
        plt.show()